# Decision Tree-Grid-Classification — Customer Purchase Prediction

## Project Overview

This notebook demonstrates an end-to-end implementation of a Decision Tree-Grid-classification model to predict whether a customer will purchase a product based on their demographic information. The project covers data preprocessing, feature engineering, model training, and performance evaluation.

## Business Problem

A company wants to identify customers who are most likely to purchase its product. Instead of targeting every customer, the company can use historical customer data—such as **Gender**, **Age**, and **Estimated Salary**—to predict whether a customer is likely to make a purchase. This enables more focused marketing campaigns, improves conversion rates, and reduces advertising costs.

## Objective

Develop a Naive Bayes classification model that predicts whether a customer will purchase a product using the following features:

* Gender
* Age
* Estimated Salary

The target variable is:

* **Purchased**

  * **0** → Customer did not purchase the product
  * **1** → Customer purchased the product

## Workflow

1. Import the required libraries.
2. Load and inspect the dataset.
3. Perform data preprocessing.
4. Remove unnecessary features (e.g., **User ID**).
5. Encode categorical features (e.g., **Gender**).
6. Split the dataset into training and testing sets.
7. Standardize the numerical features.
8. Train the Decision Tree-Grid-classification model.
9. Evaluate the model performance using the Confusion Matrix and Classification Report.


#importing the Libraies
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
#importing the Libraies
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [4]:
# Read the input data
dataset=pd.read_csv("../data/Social_Network_Ads.csv")

In [5]:
# Print the first five rows of the datasetdataset
dataset.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [6]:
# Convert the categorical columns to numerical values in the dataset.
dataset=pd.get_dummies(dataset,drop_first=True).astype(int)

In [7]:
# Drop the column "User ID" as it not required for the model.
dataset=dataset.drop("User ID",axis=1)

In [8]:
# Print the first five rows of the dataset
dataset.head()

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1


In [9]:
# Get the count of number of purchased and not purchased count in the dataset.
dataset["Purchased"].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [10]:
# Split the independent and dependant varaiables.
indep=dataset[["Age","EstimatedSalary","Gender_Male"]]
dep=dataset["Purchased"]

In [11]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [12]:
# Standard Scaler to put different features onto the same scale
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [13]:
# Import Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

In [14]:
#https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter

In [15]:
# Import GridSearchCV to automate hyperparameter tuning
from sklearn.model_selection import GridSearchCV

param_grid = {'criterion':['gini','entropy'],
              'max_features': ['auto','sqrt','log2'],
              'splitter':['best','random']} 

grid = GridSearchCV(DecisionTreeClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
# fitting the model for grid search 
grid.fit(X_train, y_train) 

Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
20 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
11 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Anaconda3\Lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Anaconda3\Lib\site-packages\sklearn\base.py", line 471, in _validate_params
    validate_parameter_constraints(
    ~~~~~~~~~~~~

,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_features': ['auto', 'sqrt', ...], 'splitter': ['best', 'random']}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [17]:
# print best parameter after tuning 
#print(grid.best_params_) 
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(X_test) 

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [18]:
# Calculate f1 score
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'criterion': 'entropy', 'max_features': 'sqrt', 'splitter': 'best'}: 0.8738789599757187


In [19]:
# Display confusion matrix
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[75 10]
 [ 7 42]]


In [20]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.91      0.88      0.90        85
           1       0.81      0.86      0.83        49

    accuracy                           0.87       134
   macro avg       0.86      0.87      0.86       134
weighted avg       0.88      0.87      0.87       134



In [22]:
# Calculate roc auc score
from sklearn.metrics import roc_auc_score

y_prob = grid.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.8697478991596639
